# Allocentric social place fields — demo

Compute occupancy-normalized rate maps of a focal animal's cells over each
(self and partner) animal's allocentric `(x, y)`, then look at a few example
cells, the population classification, and the **self-position confound control**.

See [`ephys/social_spatial_fields.py`](social_spatial_fields.py) for the API and
the project `README.md` / `CLAUDE.md` for the gotchas. Two that matter most here:

- occupancy is computed over the **target's** `(x, y)`, not the focal animal's;
- tracking↔ephys conversion lives only in
  `video.tracking_import.resolve_tracking_on_ephys_clock`
  (`MultiAnimalSession.get_tracking_on_ephys_clock` is a thin wrapper around it).

Only the focal animal needs ephys, so we load through
`ingestion.focal_session.load_focal_session_inputs` rather than a
`MultiAnimalSession` — the targets contribute tracking trajectories only.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
%matplotlib inline

from ingestion.focal_session import load_focal_session_inputs
from ephys.social_spatial_fields import (
    DEFAULT_STRATUM_RADIUS_CM,
    compare_self_stratum_control,
    compute_social_place_fields,
    modal_occupancy_center,
)
from ephys import social_spatial_plots as sp

## 1. Load a session and compute social place fields

Edit `SESSION`, `ANIMALS`, and `FOCAL` for your data. The focal animal's spikes
generate the maps; every animal in `ANIMALS` (including the focal) is a target.

`n_shuffles` is **not** a free knob. The per-cell FDR family is the targets, so
the smallest reachable q is `n_targets / (n_shuffles + 1)`; with 3–4 targets at
the default `sig_alpha=0.01` anything below ~400 shuffles makes it arithmetically
impossible for any cell to be called tuned, whatever the effect size. 500 is the
smallest safe value here — so don't cut it to save time.

Time is no longer the reason to. The null loop hoists all tracking-side work out
of the iteration and shares it across cells, and `shuffle_split_half` now
defaults to `False` (the split-half *observed* value is still reported; only its
unused shuffle p-value is skipped). Measured together at ~13× on a 7500 s
session.

In [ ]:
SESSION = '20251210'
ANIMALS = ['631', '634', '635']
FOCAL = '631'

# Only the focal animal's ephys is loaded; targets contribute tracking only.
inputs = load_focal_session_inputs(SESSION, FOCAL)

results = compute_social_place_fields(
    inputs.ks_focal, inputs.tracking, inputs.sync,
    focal_animal=FOCAL, target_animals=ANIMALS,
    pixels_per_cm=inputs.pixels_per_cm,
    bin_size_cm=5.0, smoothing_sigma_cm=5.0,
    speed_threshold_cms=5.0, speed_filter_subject='target',
    n_shuffles=500, null_method='position_shuffle',
    use_quality_cells=True,
)
results.cell_classification['category'].value_counts()

## 2. Rate-map grids for a few example cells

Top cells by max Skaggs bits/spike across targets. One panel per target animal
(the focal target is the self place field).

Each heatmap is the **target's** position, and the white overlay is the **focal**
animal's — both in the same arena coordinates, so a panel reads "target here,
focal there". Without a stratum (below) the focal ranged over the whole arena,
so the overlay is its dwell-time contours with a `+` at its modal bin, and the
title says *position unconditioned*: nothing here rules out that an apparent
partner field is really the focal animal's own place field. Pass
`show_focal=False` to drop the overlay.

In [ ]:
import importlib
importlib.reload(sp)
df = results.cell_classification
bits_cols = [f'bits_per_spike_{t}' for t in ANIMALS if f'bits_per_spike_{t}' in df.columns]
top = df.assign(_m=df[bits_cols].max(axis=1)).sort_values('_m', ascending=False)
top_clusters = top['cluster_id'].head(3).astype(int).tolist()

for cid in top_clusters:
    sp.plot_rate_maps_grid(results, cluster_id=cid)

## 3. Population classification summary

In [ ]:
sp.plot_social_place_summary(results)
# sp.plot_field_stability(results)

## 4. Self-position confound control

The nulls above break the spike↔target-position pairing *entirely*, so what they
reject is "this cell's firing is unrelated in time to the target's position". A
cell that codes only the **focal animal's own** position violates that whenever
the two animals' trajectories are correlated — which is the defining property of
a shared arena. Such a cell is reported as partner-tuned without ever having
represented the partner.

The control is to hold the focal animal still: restrict every map to samples
where it sat within `radius_cm` of one spot, so its own position has almost no
variance and cannot generate a map over the target's position.

`DEFAULT_STRATUM_RADIUS_CM` is deliberately permissive. A radius at place-field
scale (~5 cm) is the tighter control, but it retains under 1% of samples and
starves most cells below `min_n_spikes`, leaving no power to detect a real
partner field. Check `residual_spread_cm` in the diagnostics to see how loose the
control actually ended up, and tighten the radius if the retained budget allows.

In [ ]:
RADIUS_CM = DEFAULT_STRATUM_RADIUS_CM   # permissive; tighten if enough data survives

comparison = compare_self_stratum_control(
    inputs.ks_focal, inputs.tracking, inputs.sync,
    focal_animal=FOCAL, target_animals=ANIMALS,
    self_stratum_radius_cm=RADIUS_CM,
    pixels_per_cm=inputs.pixels_per_cm,
    bin_size_cm=5.0, smoothing_sigma_cm=5.0,
    speed_threshold_cms=5.0, speed_filter_subject='target',
    n_shuffles=500, use_quality_cells=True,
)

# How tight did the control actually end up, and how much data survived?
for target, d in comparison.parameters['self_stratum_diagnostics'].items():
    print(f"{target}: kept {d['n_retained']:>6} samples "
          f"({100 * d['fraction_retained']:5.2f}%), {d['retained_seconds']:7.1f} s, "
          f"focal spread {d['residual_spread_cm']:.2f} cm")

### Which partner tuning survives the control?

`lost_to_stratum` flags tests that were significant unrestricted and are not once
the focal animal is pinned — candidate self-position leakage.

Read `n_spikes_stratum` before believing it. A tight stratum starves cells whose
fields sit away from it, and a cell that vanished for want of spikes has not been
shown to be a confound. The **self** target is the positive control and is
*expected* to collapse: the focal is confined by construction.

Compare `p_full` against `p_stratum`, never `bits_full` against `bits_stratum` —
fewer retained spikes bias Skaggs upward through noise while losing real signal
pushes it down, so the two runs are not on a common scale. Each run's
permutation test is still valid, because each builds its null from its own
retained data.

In [ ]:
tbl = comparison.table
partners = tbl[~tbl['is_self']]

print(f"partner tests significant unrestricted : {int(partners['sig_full'].sum())}")
print(f"  ... still significant under stratum  : {int(partners['survives_stratum'].sum())}")
print(f"  ... lost (leakage OR spike-starved)  : {int(partners['lost_to_stratum'].sum())}")
print(f"self tests still significant (want 0)  : "
      f"{int(tbl[tbl['is_self']]['survives_stratum'].sum())}")

cols = ['cluster_id', 'target', 'p_full', 'p_stratum',
        'n_spikes_full', 'n_spikes_stratum', 'survives_stratum']
partners[partners['sig_full']].sort_values('p_stratum')[cols].head(15)

In [ ]:
# Side-by-side maps for a cell that survived. Under the stratum the overlay
# becomes the disc the focal animal was confined to, and the self panel should
# be empty outside it — the positive control, visible at a glance.
survivors = partners[partners['survives_stratum']]['cluster_id'].tolist()
cid = int(survivors[0]) if survivors else int(comparison.table['cluster_id'].iloc[0])

sp.plot_rate_maps_grid(comparison.unrestricted, cluster_id=cid)   # focal free
sp.plot_rate_maps_grid(comparison.stratified, cluster_id=cid)     # focal pinned